# Org-id analysis

This notebook runs once per day to prepare outputs (in the output folder) to be displayed in the streamlit app.

## Setup

In [1]:
#rerun if app is deleted and remade
import pyvis
import plotapi
import stvis

In [2]:
#get API key
import os
import deepnote_toolkit
deepnote_toolkit.set_integration_env()
sankey_id = os.environ.get("PLOTAPI_KEY")

In [3]:
#Import the libraries used in the notebook 
import seaborn as sns
import pandas as pd
import numpy as np
from datetime import datetime
import re

import plotapi
from plotapi import Sankey
Sankey.api_key(sankey_id)

Connect to IATI-tables to download IATI data.

In [4]:
sns.set_context('notebook')

#start noteql session
import noteql
# Restart postgres to make sure any existing connections get dropped
!sudo service postgresql restart
session = noteql.Session(datasette_url='https://datasette.codeforiati.org/iati.json', connect_args={'connect_timeout': 1000})

# Download and prepare the data

All reported participating organisations, all recipient orgs in transactions, and all provider orgs in transactions.

## iati-tables

Participating organisations, and who reported them

In [5]:
%%nql partorg=DF

SELECT DISTINCT 
  prefix,
  reportingorg_ref,
  ref as "org-id",
  type,
  typename,
  narrative,
  "part_org" as 'reftype'
FROM
  participatingorg
WHERE ref is not null AND NOT TRIM(ref)=''

Receiver and provider organisations and who reported them, from transactions.

In [6]:
%%nql reciporg=DF

SELECT DISTINCT 
    prefix,
    reportingorg_ref,
    receiverorg_ref as 'org-id',
    receiverorg_type as 'type',
    receiverorg_typename as 'typename',
    receiverorg_narrative as 'narrative',
    "receiver_org" as 'reftype'
FROM trans 
WHERE receiverorg_ref is not null AND NOT TRIM(receiverorg_ref)=''

In [7]:
%%nql provorg=DF

SELECT DISTINCT 
    prefix,
    reportingorg_ref,
    providerorg_ref as 'org-id',
    providerorg_type as 'type',
    providerorg_typename as 'typename',
    providerorg_narrative as 'narrative',
    "provider_org" as 'reftype'
FROM trans 
WHERE providerorg_ref is not null AND NOT TRIM(providerorg_ref)=''

Note - did not use planned disbursements as very few had references. Could add in future.

Receiver and provider combinations, from transactions

In [8]:
%%nql rec_prov=DF

SELECT DISTINCT
    providerorg_ref as 'prov_org',
    receiverorg_ref as 'rec_org'
FROM trans 
WHERE providerorg_narrative is not null AND NOT TRIM(providerorg_narrative)=''
     AND providerorg_ref is not null AND NOT TRIM(providerorg_ref)=''
     AND receiverorg_narrative is not null AND NOT TRIM(receiverorg_narrative)=''
     AND receiverorg_ref is not null AND NOT TRIM(receiverorg_ref)=''

## Prep IATI-tables downloads

### Full table

Combine into one table of all reported org-ids.

In [9]:
all_refs = pd.concat([partorg,reciporg,provorg]).reset_index(drop=True)

Remove language from start of narrative. For multi narratives this will still be messy, but is good enough for now.

In [10]:
all_refs['narrative'] = all_refs['narrative'].str.replace("\D{2}: ","", regex=True)

Add lowercase narratives and org-ids for filtering

In [11]:
#function to create lowercase, no whitespace columns
def prep(df_column):
    col = df_column.str.replace(" ", "").str.lower().str.strip()
    return col

In [12]:
#Add org-ids and narratives with no whitespace, newlines, or case
all_refs['org-id_lower'] = prep(all_refs['org-id'])
all_refs['narrative_lower'] = prep(all_refs['narrative'])

all_refs['concat_lower'] = all_refs['org-id_lower'] + all_refs['narrative_lower']

Add registration agency prefix column

In [13]:
all_refs['reg_agency'] = (all_refs['org-id_lower'].str.split('-', n=2, expand=True)[0] + '-' + all_refs['org-id_lower'].str.split('-', n=2, expand=True)[1])

In [14]:
all_refs

### De-duped table

Remove duplicates from combination of participating, receiver and provider org lists.

In [15]:
all_refs_filt = all_refs.drop(['reftype','type','typename'], axis=1).drop_duplicates()

In [16]:
all_refs_filt

### De-duped plus counts

Add count of org id and narrative occurrences

In [17]:
all_ref_count = all_refs_filt.copy()

#org-id + narrative occurences
all_ref_count["org-id_narrative_count"] = all_ref_count['concat_lower'].map(all_ref_count['concat_lower'].value_counts())
all_ref_count['org-id_narrative_count'] = all_ref_count['org-id_narrative_count'].map(int)

In [18]:
all_ref_count

## Download reference tables and codelists

In [19]:
#download tables
!wget https://org-id.guide/download.csv 
!wget https://iatistandard.org/reference_downloads/203/codelists/downloads/clv3/csv/en/OrganisationIdentifier.csv
!wget https://iatiregistry.org/publisher/download/csv
!wget https://codeforiati.org/gov-id-finder-data/downloads/org-ids.csv 
!wget https://codelists.codeforiati.org/api/csv/en/CRSChannelCode.csv 

#load into dataframes
#org-id.guide registration agencies
reg_agency = pd.read_csv("download.csv")
#v1 org-ids
old_orgid = pd.read_csv("OrganisationIdentifier.csv")
#registry org-ids
registry = pd.read_csv("csv")
#code4iati gov finder org-ids
gov_finder = pd.read_csv("org-ids.csv")
#dac channel codes
channel_code = pd.read_csv("CRSChannelCode.csv")

#remove downloads
!rm -rf "download.csv" "OrganisationIdentifier.csv"  "csv" "org-ids.csv" "CRSChannelCode.csv"

## Prep reference tables and codelists

Prepare org-ids from DAC codes

In [20]:
#get potential org-ids from channel codes
channel_code['code'] = channel_code['code'].astype(str)
channel_code['XMDAC'] = "XM-DAC-" + channel_code['code']

Add lowercase, no whitespace ids and narratives to codelists as needed.

In [21]:
reg_agency['code_lower'] = prep(reg_agency['code'])

old_orgid['code_lower'] = prep(old_orgid['code'])
old_orgid['name_lower'] = prep(old_orgid['name'])

registry['IATI Organisation Identifier_lower'] = prep(registry['IATI Organisation Identifier'])
registry['Publisher_lower'] = prep(registry['Publisher'])

gov_finder['org_id_lower'] = prep(gov_finder['org_id'])
gov_finder['name_lower'] = prep(gov_finder['name'])

channel_code['name_lower'] = prep(channel_code['name'])
channel_code['XMDAC_lower'] = prep(channel_code['XMDAC'])

Prepare generic XM DAC org-ids for filtering

In [22]:
refs = pd.read_csv("data/Generic_XM-DACs_Jul24.csv")

#build generic dac codes
refs['code'] = refs['code'].astype(str)
refs['XMDAC'] = "XM-DAC-" + refs['code']

# Remove blank references
refs = refs.dropna(subset=['code'])

Remove "Miscellaneous" from V1 codes

In [23]:
old_orgid = old_orgid[old_orgid['name'] != "Miscellaneous"]

## Dashboard outputs

In [24]:
!wget https://dashboard.iatistandard.org/generated/data/csv/publishers.csv

pubs = pd.read_csv("publishers.csv")

!rm -rf publishers.csv

In [25]:
#filter to orgs with at least one file, where the reigstry orgid =/= data 
filt_pubs = pubs[(pubs['Files'] > 0) & (pubs['Reporting Org on Registry'] != pubs['Reporting Orgs in Data'])].reset_index()

mismatch_orgs = filt_pubs[['Publisher Name','Reporting Org on Registry','Reporting Orgs in Data','Reporting Orgs in Data (count)']].copy()
mismatch_orgs.rename(columns={'Publisher Name': 'Reporting Org Name',
                            'Reporting Org on Registry': 'Approved Registry Org-id',
                            'Reporting Orgs in Data':'Org-id in Published Data',
                            'Reporting Orgs in Data (count)':"Org-id Count"}, inplace=True)

mismatch_orgs

In [26]:
#clean up
del pubs, filt_pubs, partorg,reciporg,provorg

# Clean the data

- Trim leading/trailing whitespace and newlines from narratives and references

- Remove obvious issues such as blank references 

- Remove all/part of the data from reporting orgs with known issues which mean a participating organisation could not be identified. 

- Remove generic XM-DAC codes.

- Remove org-ids unlikely to be valid due to their length. Currently remove any over 50 characters long.

## Clean narratives and refs 

In [27]:
#function to clean org-ids and narratives - remove new lines, whitespace
def clean_string(value):
    text = re.sub(r"\n", "", value).strip()
    text = re.sub(r"\r", "", text).strip('"').replace('•', '')
    return text

In [28]:
#strip whitespace and newlines
all_refs_filt['narrative'] = all_refs_filt['narrative'].map(clean_string)
all_refs_filt['org-id'] = all_refs_filt['org-id'].map(clean_string)

## Filter narratives

In [29]:
#known redactions/issues
#narratives
narrative_list = ['nan',',','','n/a','#N/A','EN: ','Odefinierat','EN: IP not published', 'EN: , FR: ','0'
                'IP not published','Not applicable','Confidential',
                'UK - Department for International Development (DFID)',
                'USAID redacted this field in accordance with the exceptions outlined in the Foreign Aid Transparency and Accountability Act of 2016.']

In [30]:
#clean narratives
trim_nar = all_refs_filt[~all_refs_filt['narrative'].isin(narrative_list) & ~all_refs_filt['narrative'].str.isspace()]
trim_nar = trim_nar[trim_nar['narrative'].str.contains('Private donors|Övrigt|Övriga')==False].reset_index(drop=True)

## Filter references

In [31]:
#refences to remove
#bespoke list + generic dac codes
ref_list = [' ','Not registered in IATI','0','#','','nan',',','n/a','#N/A'] + refs['code'].to_list() + refs['XMDAC'].to_list()

In [32]:
#clean references
trim_ref = trim_nar[~trim_nar['org-id'].isin(ref_list)]
trim_ref = trim_ref[trim_ref['org-id'].str.contains('XM-DAC-41122-VN-')==False]

## Length filter

In [33]:
#trim based on length
trim_length = trim_ref[trim_ref['org-id'].str.len() < 51].copy()

In [34]:
#clean up 
del ref_list

# Shared References

This section prepares data for the networked org-ids analysis

## Join codelists

Join cleaned data to codelists to facilitate filtering by reporting organisation

In [35]:
#join reporting organisation info
rep_org_join = trim_length.merge(registry, how='left',left_on='reportingorg_ref',right_on="IATI Organisation Identifier")
rep_org_join = rep_org_join[['Publisher','prefix','reportingorg_ref','Organization Type','HQ Country or Region',
                                'org-id','narrative','reg_agency']]
                                
rep_org_join.rename(columns={'Publisher': 'reportingorg_name',
                            'Organization Type': 'reportingorg_type',
                            'HQ Country or Region':'reportingorg_country'}, inplace=True)

#remove rows with incorrect reporting org refs (do not join to the registry table)
rep_org_join = rep_org_join[~rep_org_join['reportingorg_name'].isna()]                      

Join org-id info to filter invalid org-ids.

In [36]:
#join org-id information
joined_orgs = rep_org_join.merge(reg_agency, how='left',left_on='reg_agency',right_on="code_lower").merge(old_orgid, how='left',left_on='org-id',right_on="code").merge(registry, how='left',left_on='org-id',right_on="IATI Organisation Identifier")
joined_orgs = joined_orgs[['reportingorg_name','prefix','reportingorg_ref','reportingorg_type',
                            'reportingorg_country','org-id','narrative','Publisher','name/en',
                            'name']]

joined_orgs.rename(columns={'Publisher': 'registry_narrative',
                            'name/en':'reg_agency_narrative',
                            'name':'v1_orgid'}, inplace=True)
joined_orgs

## Filter data

Check that the reported organisation references:

- Are a valid reporting org ID from the registry OR

- Have a valid prefix from org-id UNLESS

- They are a v1 org id

In [37]:
filtered_orgs = joined_orgs[~joined_orgs['reg_agency_narrative'].isna() | ~joined_orgs['registry_narrative'].isna() | ~joined_orgs['v1_orgid'].isna()]
filtered_orgs

## Prep data for plotting/tables

### Chord data

In [38]:
#Get unique reporting org name and org reference combinations
df = filtered_orgs[["reportingorg_name","org-id"]].drop_duplicates()

Output a table for a flourish  plot

In [39]:
#merge to pair up organisations that report the same ref, and count 
result = df.merge(df, on='org-id').groupby(['reportingorg_name_x','reportingorg_name_y']).count().reset_index()
result.columns = ['source', 'target', 'value']

#remove self matches
mask = result.source!=result.target
chord_df = result[mask].reset_index(drop='True')

#output df that can be used for chord diagrams
chord_df = chord_df.sort_values(by="value", ascending=False)
chord_df

Additional filters for bokeh plot

In [40]:
#remove duplicate relationships
mask1 = np.logical_and(result.source<result.target, result.value>0)
bokeh_df = result[mask1].reset_index(drop='True')
bokeh_df = bokeh_df.sort_values(by="source")

Output a table for a plotAPI style plot

In [41]:
#merge to pair up organisations that report the same ref
result2 = df.merge(df, on='org-id')[['reportingorg_name_x','reportingorg_name_y']]

#remove self matches
mask2 = result2.reportingorg_name_x!=result2.reportingorg_name_y
result_masked = result2[mask2].reset_index(drop='True')

#make co-occurence dataframe
chord_df2 = result_masked.groupby(['reportingorg_name_x','reportingorg_name_y']).size().unstack().fillna(0).map(int)

#prep matrix
names = list(chord_df2.columns)
matrix = chord_df2.values.tolist() 

### Bar chart data

Common references

In [42]:
top_count = filtered_orgs[['org-id','reportingorg_ref']].drop_duplicates().groupby(['org-id']).count().sort_values(by=['reportingorg_ref'],ascending=False).reset_index().head(20)
top_count

In [43]:
most_ref = filtered_orgs[['reportingorg_name','org-id']].drop_duplicates().groupby(['reportingorg_name']).count().sort_values(by=['org-id'],ascending=False).reset_index().head(20)
most_ref

In [44]:
filtered_orgs

In [45]:
#clean up
del rep_org_join, joined_orgs, df, result, mask

# Reported Org-ids

This section prepares data for the reported org-ids analysis

## Validate the org-ids

### Valid registration agencies

Check that the organisation-identifiers are constructed using a valid registration agency list, according to https://org-id.guide/. 

In [46]:
#add "-"" to prefix for extra filter
reg_agency['codedash'] = reg_agency['code_lower'] + '-'

In [47]:
trim_agency = trim_length[(trim_length['reg_agency'].isin(reg_agency['code_lower'])) &
         (~trim_length['org-id_lower'].isin(reg_agency['code_lower'])) &
         (~trim_length['org-id_lower'].isin(reg_agency['codedash']))].copy()

trim_agency['Source'] = 'Community sourced org-id'

## Source other valid org-ids 

### Prep sources

In [48]:
#Registry IDs
#Source: https://iatiregistry.org/publisher/

reg_ids = registry[['IATI Organisation Identifier','Publisher']].copy()
reg_ids.rename(columns={'IATI Organisation Identifier': 'org-id', 'Publisher': 'narrative'}, inplace=True)
reg_ids['Source'] = 'Registry org-id'
reg_ids

In [49]:
#V1 IDs
#Source: https://iatistandard.org/en/iati-standard/203/codelists/organisationidentifier/ 

v1_ids = old_orgid[['code','name']].copy()
v1_ids.rename(columns={'code': 'org-id', 'name': 'narrative'}, inplace=True)
v1_ids['Source'] = 'V1 org-id'
v1_ids

In [50]:
#Government Organisation IDs
#Source: https://gov-id-finder.codeforiati.org/ 

#remove prefixes with no code
gov_finder = gov_finder[gov_finder['org_id'].str.len()>7]

gov_ids = gov_finder[['org_id','name']].copy()

gov_ids.rename(columns={'org_id': 'org-id','name': 'narrative'}, inplace=True)
gov_ids['Source'] = 'Government org-id finder'
gov_ids

In [51]:
#OECD Channel Codes
#Source: https://codelists.codeforiati.org/CRSChannelCode/ 
#Note - the direct OECD download format is not well formatted for this use.

#drop withdrawn codes
channel_code = channel_code[channel_code['status'] == 'active']

#drop generic codes
channel_code = channel_code[~channel_code['XMDAC'].isin(refs['XMDAC'])]

dac_ids = channel_code[['XMDAC','name']].copy()
dac_ids.rename(columns={'XMDAC': 'org-id','name': 'narrative'}, inplace=True)
dac_ids['Source'] = 'OECD Channel codes'
dac_ids

###  Combine other sources

In [52]:
other_sources = pd.concat([reg_ids,v1_ids,dac_ids,gov_ids]).reset_index(drop=True)

#add lower columns
other_sources['org-id_lower'] = prep(other_sources['org-id'])
other_sources['narrative_lower'] = prep(other_sources['narrative'])

other_sources

### Filter validated list

If a reported narrative org-id exists in the other sources, prioritise them.

In [53]:
trim_source = trim_agency[(~trim_agency['narrative_lower'].isin(other_sources['narrative_lower'])) &
                (~trim_agency['org-id_lower'].isin(other_sources['org-id_lower']))]

## Combine all sources

Combine the sources with the filtered list, remove any duplicates.

In [54]:
#concat in order of priority
filtered = pd.concat([other_sources,trim_source])

#drop extra columns
filtered = filtered.drop(['prefix','reportingorg_ref','reg_agency'], axis=1)

#add missing lower columns
filtered['concat_lower'] = filtered['org-id_lower'] + filtered['narrative_lower']

In [55]:
#deduplicate, keep first
filtered_dedup = filtered.drop_duplicates(subset=['concat_lower'], keep='first').copy()
filtered_dedup = filtered_dedup.drop(['narrative_lower','org-id_lower'], axis=1)

### Select the narrative

Select the most common narrative for any repeated org-ids

In [56]:
filtered_count = filtered_dedup.merge(all_ref_count[['org-id_narrative_count','concat_lower']].drop_duplicates(), 
                                    how='left',on=['concat_lower'])
filtered_count = filtered_count.drop(['concat_lower'], axis=1)


#group and order, keep head
filt_narrative = filtered_count.sort_values(['org-id_narrative_count'],ascending=False).groupby('org-id').head(1).reset_index(drop=True)
filt_narrative.rename(columns={'narrative': 'Most Common Narrative',"org-id_narrative_count":"Organisations using Reference"}, 
                    inplace=True)

In [57]:
#add registration agency column
filt_narrative['org-id_prefix'] = (filt_narrative['org-id'].str.split('-', n=2, expand=True)[0] + '-' + filt_narrative['org-id'].str.split('-', n=2, expand=True)[1])
filt_narrative['code_lower'] = prep(filt_narrative['org-id_prefix'])

#Join on org-id lists for countries
reporgs = filt_narrative.merge(reg_agency[['code','coverage','code_lower']],how='left',on='code_lower')
reporgs = reporgs.drop(['code_lower','org-id_prefix'], axis=1)
reporgs.rename(columns={'coverage': 'Reg Agency Country','code':'Reg Agency Code'}, inplace=True)
reporgs = reporgs[['org-id','Most Common Narrative','Organisations using Reference','Source','Reg Agency Code','Reg Agency Country']]

In [58]:
#add column with link to d-portal
reporgs['d-portal link'] = "https://d-portal.iatistandard.org/ctrack.html?/participating-org@ref=" + reporgs['org-id'] + "#view=main"

In [59]:
reporgs

## Plots

### Sankey

- Initial unique count - all unique org-id and narrative combinations

- Cleaned a) narratives b) references c) overlong refs

- Validated reg agency

- Other sources a) registry ids b) v1 refs c) code4iati gov id finder, d) oecd channel codes 

- Combined validated lists, in order of priority - Registry, dac, v1, gov finder, community

Sankey source, target and value

In [60]:
sankey = pd.DataFrame({'source':['IATI Data','IATI Data','IATI Data','IATI Data',
                        'Cleaned References','Cleaned References','Validated org-ids',
                        'IATI Registry','V1 References','OECD Codes',
                        'Gov-id finder','Combined Sources','Combined Sources',
                        'Unique References','Unique References' ],
        'target':['Filtered Narratives','Filtered org-ids','Overlong org-ids','Cleaned References',
                    'Invalid org-ids','Validated org-ids','Combined Sources',
                    'Combined Sources','Combined Sources','Combined Sources',
                    'Combined Sources','Duplicated References','Unique References',
                    'Duplicated org-ids','FINAL REFERENCE LIST'],
        'value':[all_refs_filt['concat_lower'].nunique() - trim_nar['concat_lower'].nunique(),
                (trim_nar['concat_lower'].nunique()- trim_ref['concat_lower'].nunique()),
                (trim_ref['concat_lower'].nunique()- trim_length['concat_lower'].nunique()),
                trim_length['concat_lower'].nunique(),
                (trim_length['concat_lower'].nunique() - trim_agency['concat_lower'].nunique()),
                trim_agency['concat_lower'].nunique(),
                trim_agency['concat_lower'].nunique(),
                len(reg_ids),
                len(v1_ids),
                len(dac_ids),
                len(gov_ids),
                trim_agency['concat_lower'].nunique() - trim_source['concat_lower'].nunique() +  (filtered['concat_lower'].nunique() - len(filtered_dedup)),
                len(filtered_dedup),
                len(filtered_dedup) - len(reporgs),
                len(reporgs)],
        'description':[
                "References removed by narrative cleaning",
                "References removed by org-id cleaning",
                "References removed due to overlong org-ids",
                'References remaining after cleaning',
                "References removed by registration agency validation",
                "References remaining after registration agency validation",
                'Cleaned and validated references from IATI data',
                "References from the IATI registry",
                "References from V1 of the IATI standard",
                "References from OECD DAC Channel Codes",
                "References  from the Code4IATI Goverment ID finder",
                "References removed to prioritise existing validated references",
                'Unique references from all sources',
                'Narratives removed to deduplicate org-ids',
                'Final list of references'
                ]})
sankey

Sankey node data

In [61]:
nodes = [
    {"name": "IATI Data", "description": "All unique references (org-id and narrative combinations) in IATI data. Taken from the participating-org, receiver-org, and provider-org elements."},
    {"name": "Filtered Narratives", "description": "Blank, generic, and redacted narratives"},
    {"name": "Filtered org-ids", "description": "Blank, generic, and known internal org-ids"},
    {"name": "Overlong org-ids", "description": "Overlong org-ids of more than 50 characters"},
    {"name": "Cleaned References", "description": "Remaining references following cleaning steps"},
    {"name": "IATI Registry", "description":"Reporting organisation references from the IATI Registry"},
    {"name": "V1 References", "description":"Organisation references from V1 of the IATI Standard"},
    {"name": "OECD Codes", "description":"References constructed from organisations OECD DAC Channel codes"},
    {"name": "Gov-id finder", "description":"References from the Code4IATI Government Organisation ID Finder"},
    {"name": "Invalid org-ids", "description":"Org-ids which do not follow the format {RegistrationAgency}-{RegistrationNumber}, where {RegistrationAgency} is a valid Registration Agency"},
    {"name": "Validated org-ids", "description":"Org-ids constructed using a valid Registration Agency code"},
    {"name":"Combined Sources","description":"Cleaned and validated references from all sources"},
    {"name":"Duplicated References","description":"If an org-id or narrative from a validated source was duplicated, the validated source was prioritised in the following order: IATI Registry, OECD Codes, V1 References, Gov-id finder"},
    {"name":"Duplicated org-ids","description":"If multiple narratives were reported for the same org-id, the most common narrative was retained"},
]

Build sankey

In [62]:
sankey_dict = sankey.to_dict('records') 

In [63]:
#format and save plot
Sankey(sankey_dict,
    noun="references",
    nodes=nodes,
    link_verb="",
    node_verb="",
    width = 1050,
    height = 600,
    node_alignment = 'left'
    ).to_html('outputs/sankey.html')

### All possible narratives

All possible narratives attached to an org-id for search box

In [64]:
#combine iati data and other sources
all_ref_count['Source'] = 'Community sourced org-id'
all_narratives = pd.concat([all_ref_count[['org-id','narrative','Source','org-id_lower','narrative_lower']],other_sources[['org-id','narrative','Source','org-id_lower','narrative_lower']]])
all_narratives = all_narratives.fillna('').drop_duplicates()
all_narratives

In [65]:
#clean up
del trim_length, reg_ids, v1_ids, gov_ids, dac_ids, filtered_count, filt_narrative

# Receiver Provider Network

Pull out all receiver and provider links.

In [66]:
rec_prov

Add most common referenced org name from "the list" for each org

In [67]:
prov_names = rec_prov.merge(reporgs,how='left', left_on ='prov_org', right_on='org-id')[['Most Common Narrative','prov_org','rec_org']]

#remove rows with invalid org refs (do not join to the list)
prov_names = prov_names[~prov_names['Most Common Narrative'].isna()]

#rename
prov_names.rename(columns={'Most Common Narrative':'Provider'},inplace=True)

In [68]:
all_links_list = prov_names.merge(reporgs,how='left', left_on ='rec_org', right_on='org-id')[['Provider','prov_org','Most Common Narrative','rec_org']]

#remove rows with invalid org refs (do not join to the list)
all_links_list = all_links_list[~all_links_list['Most Common Narrative'].isna()]

#drop dups
all_links_list = all_links_list.drop_duplicates()

#rename
all_links_list.rename(columns={'Most Common Narrative':'Receiver'},inplace=True)

In [69]:
#add org-id to narrative for search bar
all_links_list['prov_string'] = all_links_list['Provider'] + " (" + all_links_list['prov_org'] + ")"
all_links_list['rec_string'] = all_links_list['Receiver'] + " (" + all_links_list['rec_org'] + ")"

In [70]:
all_links_list

In [71]:
all_provs = all_links_list[['Provider','prov_org','prov_string']].copy()
all_provs.rename(columns={'Provider':'Narrative','prov_org':'orgid','prov_string':'string'},inplace=True)

all_recs = all_links_list[['Receiver','rec_org','rec_string']].copy()
all_recs.rename(columns={'Receiver':'Narrative','rec_org':'orgid','rec_string':'string'},inplace=True)

all_references = pd.concat([all_provs, all_recs], ignore_index=True).drop_duplicates()
all_references

In [72]:
#clean up
del all_provs, all_recs

# Country Sector search

Downloading the default table

## Pull from d-query

In [73]:
country = 'AF'
sector = '72012'
url = "http://d-portal.iatistandard.org/dquery?form=csv&human=1&sql=SELECT%20DISTINCT%0Axson.aid%2C%0Axson%20%20-%3E%20%27%2Fnarrative%27-%3E0-%3E%3E%27%27%20as%20%22Participating%20Org%22%20%2C%0Axson%20%20-%3E%3E%20%27%40ref%27%20as%20%22ref%22%2C%0Axson%20%20-%3E%3E%20%27%40role%27%20as%20%22role%22%2C%0Axson%20%20-%3E%3E%20%27%40type%27%20as%20%22type%22%0AFROM%20xson%0AJOIN%20trans%20on%20xson.aid%20%3D%20trans.aid%0AWHERE%20%0Aroot%3D%27%2Fiati-activities%2Fiati-activity%2Fparticipating-org%27%20AND%0Atrans_country%20%3D%20%27" + country +"%27%20AND%20%0Atrans_sector%20%3D%27"+ sector +"%27%20AND%0Axson%20%20-%3E%3E%20%27%40role%27%20IN%20(%271%27%2C%272%27%2C%273%27%2C%274%27)%0A%3B%0A%0A"

In [74]:
dportal_data = pd.read_csv(url)

Convert role code to names

In [75]:
#convert role codes
def role_codes(value):
    if value == 1:
        return 'Funding'
    elif value == 2:
        return 'Accountable'
    elif value == 3:
        return 'Extending'
    elif value == 4: 
        return 'Implementing'
    else:
        return ''

In [76]:
dportal_data['rolename'] = dportal_data['role'].map(role_codes)

#get activity ID
dportal_data['iati-activity'] = dportal_data['aid'].str.split('=', n=2, expand=True)[1]

In [77]:
print("There are", dportal_data['iati-activity'].nunique(), "activities")

Count number of activities per reference

In [78]:
act_count = dportal_data[['Participating Org','ref','iati-activity']].groupby(['Participating Org','ref'])['iati-activity'].nunique().reset_index()
act_count.rename(columns={'iati-activity':'Activity Count'},inplace=True)
act_count['Activity Count'] = act_count['Activity Count'].astype(str)
act_count

## Use "list" to fill in missing refs/orgs

Also join act count

In [79]:
org_narratives =  dportal_data.merge(act_count, how='left', on = ['Participating Org','ref']).merge(reporgs, how='left',left_on='ref',right_on="org-id")[['ref','Participating Org','Activity Count','Most Common Narrative','Organisations using Reference','Source','rolename']].drop_duplicates()

#rename
org_narratives.rename(columns={'ref':'org-id',
        'Participating Org':'Participating Org Narrative',
        'Most Common Narrative':'List Narrative',
        'Organisations using Reference':'List - Orgs using Reference',
        'Source':'List Narrative Source',
        'rolename':'Roles'},inplace=True)

## Output overlap

In [80]:
sector_country = org_narratives[~org_narratives['List Narrative'].isna()] 
#group roles
sector_country = sector_country.groupby(['org-id','Participating Org Narrative','List Narrative','List - Orgs using Reference','List Narrative Source','Activity Count'])['Roles'].apply(','.join).reset_index()
sector_country = sector_country[['org-id','Participating Org Narrative','Activity Count','Roles','List Narrative','List - Orgs using Reference','List Narrative Source']]

#add column with link to d-portal
sector_country['d-portal link'] = "https://d-portal.iatistandard.org/ctrack.html?&country_code="+country+"&sector_code=" +sector+ "&/participating-org@ref=" + sector_country['org-id'] + "#view=main"
sector_country

# Headline figures

How many orgs are referenced in IATI data (usefully)?

How many reporting orgs do these references come from?

How many orgs don't have a reference?

In [81]:
ref_filter = ['nan','n/a','#N/A','Not registered in IATI','0','#','Not applicable','Confidential','0']
ref_filter

In [82]:
narrative_list

In [83]:
%%nql org_noref=DF

SELECT DISTINCT 
  narrative,
  ref
FROM
  (
    SELECT DISTINCT 
      receiverorg_narrative as 'narrative',
      receiverorg_ref as 'ref'
    FROM
      trans
    WHERE
      receiverorg_ref is null
      OR TRIM(receiverorg_ref) = '' 
      OR receiverorg_ref in {{ ref_filter | inclause }}
    UNION ALL
    SELECT DISTINCT 
      providerorg_narrative as 'narrative',
      providerorg_ref as 'ref'
    FROM
      trans
    WHERE
      providerorg_ref is null
      OR TRIM(providerorg_ref) = ''
      OR providerorg_ref in {{ ref_filter | inclause }}
    UNION ALL
    SELECT DISTINCT 
      narrative,
      ref
    FROM
      participatingorg
    WHERE
      ref is null
      OR TRIM(ref)=''
      OR ref in {{ ref_filter | inclause }}
  )
WHERE narrative is not null AND NOT TRIM(narrative)='' AND NOT narrative in {{ narrative_list | inclause }}

In [84]:
org_noref

In [85]:
org_noref[org_noref['narrative'].str.contains("cdi",na=False,case=False)]

In [86]:
headlines = pd.DataFrame({'headline':['orgs_referenced','reporting_orgs','no_ref'],
                        'figure':[len(reporgs),trim_agency['reportingorg_ref'].nunique(),
                        org_noref['narrative'].nunique()]})

headlines

# Save outputs

In [87]:
now = datetime.now()
today = now.strftime("%d-%m-%Y")
print("Last run completed on:", today)

In [88]:
!rm -rf "outputs/full_list.csv" "outputs/date.txt" "outputs/sector_country.csv" "outputs/most_ref.csv" "outputs/headlines.csv" "outputs/top_count.csv" "outputs/all_refs.csv" "outputs/reporgs.csv" "outputs/all_nodes.csv" "outputs/all_links_list.csv" "outputs/filtered_orgs.csv" 'mismatch_orgs.csv' 'all_narratives.csv'

with open("outputs/date.txt", "w") as text_file:
    text_file.write(today)

#final ref list
reporgs.to_csv("outputs/reporgs.csv",index=False)
#list for chords
filtered_orgs.to_csv("outputs/filtered_orgs.csv",index=False)
#bar charts
top_count.to_csv("outputs/top_count.csv", index=False)
most_ref.to_csv("outputs/most_ref.csv", index=False)
#dashboard data
mismatch_orgs.to_csv("outputs/mismatch_orgs.csv",index=False)
#all ref narratives to search
all_narratives.to_csv("outputs/all_narratives.csv", index=False)
#reciever provider list
all_links_list.to_csv("outputs/all_links_list.csv",index=False)
#all refs in receiver provider for search
all_references.to_csv("outputs/all_refs.csv",index=False)
#headline figuires (unused)
headlines.to_csv("outputs/headlines.csv",index=False)
#default sector country values
sector_country.to_csv("outputs/sector_country.csv",index=False)
#list of all refs plus reporting org
all_refs_filt.to_csv("outputs/full_list.csv",index=False)

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=710ae5c2-768a-4825-91df-2032e21c91b0' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>